# Multimodal Clinical AI — Kaggle GPU Training

**Setup (one time):**
1. New Notebook → Settings → Accelerator = `GPU T4 x2` (or P100).
2. Add Data → search **CheXpert Small** (e.g. `ashery/chexpert`) → Add.
3. (Optional) Upload this repo as a Kaggle Dataset and add it too — or use the `git clone` cell below.
4. Run all cells.

**What this notebook does:**
- Clones the repo (or uses the dataset version)
- Installs deps
- Builds train/val splits from the CheXpert CSV
- Trains the full multimodal model on CheXpert images + synthetic notes/vitals
- Saves the best checkpoint to `/kaggle/working/checkpoints/`
- Runs evaluation + a Grad-CAM demo

> **Note on real notes/vitals:** Replace `synthetic_text=True, synthetic_vitals_flag=True` in the dataset config below with paths to your MIMIC-III preprocessed files once your CITI/PhysioNet access comes through.

In [ ]:
# --- Clone repo (skip if already added as a Kaggle Dataset) ---
import os, sys
REPO = '/kaggle/working/multimodal'
if not os.path.exists(REPO):
    # OPTION A: clone from GitHub
    # !git clone https://github.com/<your-user>/<your-repo>.git {REPO}
    # OPTION B: copy from Kaggle Dataset input (recommended for private code)
    import shutil
    src = '/kaggle/input/multimodal-clinical-ai-code'  # rename if you uploaded differently
    if os.path.exists(src):
        shutil.copytree(src, REPO)
    else:
        raise FileNotFoundError('Provide repo via GitHub clone or Kaggle Dataset')
sys.path.insert(0, REPO)
print('repo:', REPO)

In [ ]:
# --- Install deps not preinstalled on Kaggle ---
!pip install -q timm==0.9.16 transformers==4.41.0 captum shap einops pydicom

In [ ]:
# --- Locate CheXpert dataset on Kaggle ---
import glob, os
candidates = glob.glob('/kaggle/input/*/CheXpert*') + glob.glob('/kaggle/input/*/train.csv')
print('Found:')
for c in candidates[:20]:
    print(' ', c)

# Adjust these to match your attached dataset
CHEXPERT_ROOT = '/kaggle/input/chexpert/CheXpert-v1.0-small'
TRAIN_CSV     = os.path.join(CHEXPERT_ROOT, 'train.csv')
VALID_CSV     = os.path.join(CHEXPERT_ROOT, 'valid.csv')
IMAGE_ROOT    = '/kaggle/input/chexpert'  # path that prepends the CSV's 'Path' column

assert os.path.exists(TRAIN_CSV), f'Missing {TRAIN_CSV} — update paths in this cell.'
print('OK')

In [ ]:
# --- Quick subsample for fast first run (remove for full training) ---
import pandas as pd
df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)
print('train', len(df_train), 'valid', len(df_valid))

SUBSAMPLE = 20000  # set None for full data
if SUBSAMPLE and len(df_train) > SUBSAMPLE:
    df_train = df_train.sample(SUBSAMPLE, random_state=42).reset_index(drop=True)
print('subsampled train:', len(df_train))

In [ ]:
# --- Build dataloaders ---
import torch
from torch.utils.data import DataLoader
from src.data.dataset import MultimodalDataset
from src.utils.config import load_yaml

cfg = load_yaml(os.path.join(REPO, 'configs/base.yaml'))

train_ds = MultimodalDataset(
    labels_csv=df_train, image_root=IMAGE_ROOT, split='train',
    uncertainty_policy=cfg['data']['uncertainty_policy'],
    u_ones_labels=cfg['data']['uncertainty_u_ones'],
)
valid_ds = MultimodalDataset(
    labels_csv=df_valid, image_root=IMAGE_ROOT, split='val',
    uncertainty_policy=cfg['data']['uncertainty_policy'],
    u_ones_labels=cfg['data']['uncertainty_u_ones'],
)
print('len train ds:', len(train_ds), 'val ds:', len(valid_ds))

BS = 32
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

# Sanity check
batch = next(iter(train_loader))
print({k: tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__ for k, v in batch.items()})

In [ ]:
# --- Build model ---
from src.models.multimodal_model import MultimodalClinicalModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
model = MultimodalClinicalModel(
    num_labels=14, d_model=512, num_heads=8,
    fusion='cross_attention',
    image_pretrained=True,  # downloads ImageNet ViT once
).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M params')

In [ ]:
# --- Train ---
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from pathlib import Path
from src.evaluate import evaluate

EPOCHS = 5  # bump up for full training
LR = 1e-4
WD = 1e-2
AMP = True

optim_ = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = CosineAnnealingLR(optim_, T_max=EPOCHS)
crit = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=(device=='cuda' and AMP))

ckpt_dir = Path('/kaggle/working/checkpoints/all__cross_attention')
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_auc = -1

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        image = batch['image'].to(device, non_blocking=True)
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attn = batch['attention_mask'].to(device, non_blocking=True)
        vitals = batch['vitals'].to(device, non_blocking=True)
        targets = batch['labels'].to(device, non_blocking=True)

        optim_.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device=='cuda' and AMP)):
            logits = model(image=image, input_ids=input_ids, attention_mask=attn, vitals=vitals)
            loss = crit(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optim_)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim_); scaler.update()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    sched.step()

    metrics = evaluate(model, valid_loader, device)
    print(f'epoch {epoch+1}: macro_AUC={metrics["macro_AUC"]} macro_AP={metrics["macro_AP"]}')

    if metrics['macro_AUC'] > best_auc:
        best_auc = metrics['macro_AUC']
        torch.save({
            'epoch': epoch+1, 'model_state_dict': model.state_dict(),
            'macro_auc': best_auc, 'config': dict(cfg),
        }, ckpt_dir / 'best_model.pt')
        print('  ✓ saved best:', best_auc)

print('done. best AUC =', best_auc)

In [ ]:
# --- Per-label AUC ---
import pandas as pd
metrics = evaluate(model, valid_loader, device)
rows = [(k.split('/', 1)[1], v) for k, v in metrics.items() if k.startswith('auc/')]
pd.DataFrame(rows, columns=['Label', 'AUC']).sort_values('AUC', ascending=False)

In [ ]:
# --- Grad-CAM demo on a single validation image ---
import matplotlib.pyplot as plt
import numpy as np
from src.xai.gradcam import ViTGradCAM, overlay_heatmap
from src.data.image_transforms import denormalize
from src.utils.labels import LABEL_NAMES

sample = valid_ds[0]
image = sample['image'].unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.sigmoid(model(image=image)).squeeze(0).cpu().numpy()
top_idx = int(np.argmax(probs))
print('Top prediction:', LABEL_NAMES[top_idx], '%.3f' % probs[top_idx])

cam = ViTGradCAM(model)
heatmap = cam.generate(image, class_idx=top_idx); cam.close()

img_rgb = (denormalize(sample['image']).permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
overlay = overlay_heatmap(img_rgb, heatmap, alpha=0.45)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(overlay); axes[1].set_title(f'Grad-CAM · {LABEL_NAMES[top_idx]}'); axes[1].axis('off')
plt.tight_layout()

In [ ]:
# --- Vitals attention rollout demo ---
from src.xai.attention_rollout import get_vitals_importance, plot_vitals_attention
vit_t = sample['vitals'].unsqueeze(0).to(device)
imp = get_vitals_importance(model, vit_t)
plot_vitals_attention(imp, sample['vitals'].cpu().numpy()); plt.show()

## Next steps
- Increase `EPOCHS`, remove `SUBSAMPLE` for full training (T4 ≈ 4–6 hr/epoch on full CheXpert).
- Run all 8 ablation configurations by setting `fusion='early'` and varying which modalities you pass to `model(...)`.
- Replace synthetic notes/vitals with MIMIC-III paired data when access arrives:
  - Add a `notes_col` to your merged CSV and pass it to `MultimodalDataset`.
  - Save preprocessed vitals as `.npy` arrays in `vitals_dir/{row_idx}.npy`.
- Push the best checkpoint to a Kaggle Dataset, attach it to the FastAPI service, and run the dashboard locally.